In [2]:
import requests
 
URL = ("https://geoportal.valencia.es/server/rest/services/"
       "OPENDATA/Trafico/MapServer/228/query")
params = {"where": "1=1", "outFields": "*", "f": "geojson",
          "outSR": 4326, "resultRecordCount": 3}
 
r = requests.get(URL, params=params, timeout=30)
print(r.status_code)
datos = r.json()
print(type(datos))
print(list(datos.keys()))

200
<class 'dict'>
['type', 'features', 'exceededTransferLimit']


In [3]:
estaciones = datos["features"]
print(len(estaciones), "estaciones")
 
primera = estaciones[0]
print(primera["properties"]["address"])
print(primera["properties"]["available"], "bicis disponibles")
print(primera["geometry"]["coordinates"])

3 estaciones
C/GUILLEM DE CASTRO esquina con C/NA JORDANA
5 bicis disponibles
[-0.38292927973315827, 39.48004223020643]


In [4]:
r = requests.get(URL, params=params, timeout=30)
r.raise_for_status()         # excepción si el código es 4xx o 5xx
datos = r.json()
if "error" in datos:         # ArcGIS avisa de errores dentro del JSON
    raise RuntimeError(datos["error"])
print("OK:", len(datos["features"]), "estaciones")

OK: 3 estaciones


In [5]:
try:
    requests.get("https://geoportal.valencia.ess/", timeout=5)
except requests.exceptions.ConnectionError as e:
    print("No se pudo conectar:", type(e).__name__)

No se pudo conectar: ConnectionError


In [6]:
def descargar_todo(where="1=1", tam_pagina=1000):
    todas, offset = [], 0
    while True:
        p = {**params, "where": where, "resultOffset": offset,
             "resultRecordCount": tam_pagina}
        r = requests.get(URL, params=p, timeout=30)
        r.raise_for_status()
        pagina = r.json()
        todas += pagina["features"]
        print(f"offset {offset}: {len(pagina['features'])} estaciones")
        if not pagina.get("exceededTransferLimit"):
            break
        offset += tam_pagina
    return todas
 
estaciones = descargar_todo(where="number <= 3", tam_pagina=2)
print("Total:", len(estaciones))

offset 0: 2 estaciones
offset 2: 1 estaciones
Total: 3


In [7]:
import pandas as pd
import textwrap
 
df = pd.json_normalize(estaciones)
print(df.shape)
print(textwrap.fill(", ".join(df.columns), width=48))

(3, 15)
type, id, geometry.type, geometry.coordinates,
properties.gid, properties.name,
properties.number, properties.address,
properties.open, properties.available,
properties.free, properties.total,
properties.ticket, properties.updated_at,
properties.update_jcd


In [8]:
df.columns = df.columns.str.replace("properties.", "")
df["lon"] = df["geometry.coordinates"].str[0]
df["lat"] = df["geometry.coordinates"].str[1]
df["actualizado"] = pd.to_datetime(df["update_jcd"], unit="ms", utc=True)
df["open"] = df["open"].map({"T": True, "F": False})
df["pct_libres"] = (df["free"] / df["total"] * 100).round(1)
print(df[["number", "actualizado", "open", "pct_libres"]])

   number               actualizado  open  pct_libres
0       1 2026-11-13 21:00:48+00:00  True        80.0
1       2 2026-11-13 21:00:19+00:00  True        53.3
2       3 2026-11-13 20:55:03+00:00  True        65.0
